# Data Cleaning: Convert Range/Object Columns to Numerical

In [57]:
import pandas as pd
import numpy as np

df = pd.read_csv('rawdataset.csv')

# Strip leading/trailing whitespace from column names
df.columns = df.columns.str.strip()

print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nData Types:\n', df.dtypes)

Shape: (103, 21)

Columns: ['Timestamp', 'Gender', 'Course', 'Study Year', 'CGPA', 'Study Days per Week', 'Study Hours per Day', 'Study Hours Before Exam', 'Study Method', 'Study Schedule', 'Follow Schedule Frequency', 'Procrastination Level', 'Deadline Management', 'Social Media During Study', 'Part-Time Job', 'Work Hours per Week', 'Study Place', 'Family Support', 'Sleep Hours', 'Stress Level (Beginning of Semester)', 'Stress Level (End of Semester)']

Data Types:
 Timestamp                               object
Gender                                  object
Course                                  object
Study Year                              object
CGPA                                    object
Study Days per Week                      int64
Study Hours per Day                     object
Study Hours Before Exam                 object
Study Method                            object
Study Schedule                          object
Follow Schedule Frequency               object
Procrastina

In [58]:
df.head()

,Timestamp,Gender,Course,Study Year,CGPA,Study Days per Week,Study Hours per Day,Study Hours Before Exam,Study Method,Study Schedule,...,Procrastination Level,Deadline Management,Social Media During Study,Part-Time Job,Work Hours per Week,Study Place,Family Support,Sleep Hours,Stress Level (Beginning of Semester),Stress Level (End of Semester)
0,3/23/2026 0:38:11,Male,Bachelor of Education,Year 2,3.00 – 3.99,6,5 – 6 hours,3 – 4 hours,Summarizing/making notes,Yes,...,2,Good,Sometimes,No,NaN,"Library, Room, Campus Study Area",Very Supportive,6 – 7 hours,2,4
1,3/23/2026 0:38:59,Female,Diploma in Computer Science,Year 3,3.00 – 3.99,5,3 – 4 hours,More than 6 hours,Doing practice questions,No,...,3,Average,Always,No,NaN,"Library, Room",Very Supportive,4 – 5 hours,2,4
2,3/23/2026 12:22:45,Female,Diploma in Science,Year 2,3.00 – 3.99,7,5 – 6 hours,More than 6 hours,Doing practice questions,Yes,...,2,Good,Sometimes,No,NaN,Room,Very Supportive,4 – 5 hours,3,4
3,3/23/2026 13:34:35,Female,Bachelor of Information Technology,Year 2,3.00 – 3.99,6,5 – 6 hours,3 – 4 hours,Summarizing/making notes,Yes,...,3,Average,Always,No,NaN,Room,Very Supportive,4 – 5 hours,2,4
4,3/23/2026 15:09:20,Female,Bachelor of Civil Engineering,Year 2,3.00 – 3.99,3,3 – 4 hours,5 – 6 hours,Doing practice questions,No,...,3,Good,Sometimes,No,NaN,"Room, Campus Study Area",Neutral,Less than 4 hours,3,4


## 1. CGPA — Convert range strings to midpoint float

In [59]:
print('Unique CGPA values:', df['CGPA'].unique())

Unique CGPA values: ['3.00 – 3.99' '4' '2.00 – 2.99' 'Below 2.00']


In [60]:
cgpa_map = {
    'Below 2.00'  : 1.00,   # midpoint of 0.00–1.99 approximated as 1.00
    '2.00 – 2.99' : 2.50,
    '3.00 – 3.99' : 3.50,
    '4'           : 4.00,   # exact value
}

df['CGPA'] = df['CGPA'].astype(str).str.strip()

# Also handle entries already entered as plain numbers (e.g. '4')
def convert_cgpa(val):
    val = str(val).strip()
    if val in cgpa_map:
        return cgpa_map[val]
    try:
        return float(val)     # already a number
    except ValueError:
        return np.nan

df['CGPA'] = df['CGPA'].apply(convert_cgpa).astype(float)
print(df['CGPA'].value_counts())
print('dtype:', df['CGPA'].dtype)

CGPA
3.5    94
2.5     5
4.0     3
1.0     1
Name: count, dtype: int64
dtype: float64


## 2. Study Year — Extract year number as integer

In [61]:
print('Unique Study Year values:', df['Study Year'].unique())

Unique Study Year values: ['Year 2' 'Year 3' 'Year 1' 'Year 5']


In [62]:
# 'Year 1' -> 1, 'Year 2' -> 2, etc.
df['Study Year'] = (
    df['Study Year']
    .astype(str)
    .str.strip()
    .str.extract(r'(\d+)')[0]   # pull the digit(s)
    .astype(float)               # float first to handle NaN safely
    .astype('Int64')             # nullable integer
)
print(df['Study Year'].value_counts())
print('dtype:', df['Study Year'].dtype)

Study Year
2    63
1    30
3     9
5     1
Name: count, dtype: Int64
dtype: Int64


## 3. Study Days per Week — Already numeric, coerce to int

In [63]:
print('Unique Study Days per Week values:', df['Study Days per Week'].unique())

Unique Study Days per Week values: [6 5 7 3 1 4 2]


In [64]:
df['Study Days per Week'] = pd.to_numeric(df['Study Days per Week'], errors='coerce').astype('Int64')
print(df['Study Days per Week'].value_counts())
print('dtype:', df['Study Days per Week'].dtype)

Study Days per Week
5    29
3    24
4    16
7    11
6    11
2     8
1     4
Name: count, dtype: Int64
dtype: Int64


## 4. Study Hours per Day — Convert range strings to midpoint float

In [65]:
print('Unique Study Hours per Day values:', df['Study Hours per Day'].unique())

Unique Study Hours per Day values: ['5 – 6 hours' '3 – 4 hours' '1 – 2 hours' 'Less than 1 hour']


In [66]:
hours_per_day_map = {
    'Less than 1 hour' : 0.5,
    '1 – 2 hours'      : 1.5,
    '3 – 4 hours'      : 3.5,
    '5 – 6 hours'      : 5.5,
    'More than 6 hours': 7.0,   # conservative estimate
}

df['Study Hours per Day'] = (
    df['Study Hours per Day']
    .astype(str).str.strip()
    .map(hours_per_day_map)
    .astype(float)
)
print(df['Study Hours per Day'].value_counts())
print('dtype:', df['Study Hours per Day'].dtype)

Study Hours per Day
3.5    40
1.5    37
5.5    21
0.5     5
Name: count, dtype: int64
dtype: float64


## 5. Study Hours Before Exam — Convert range strings to midpoint float

In [67]:
print('Unique Study Hours Before Exam values:', df['Study Hours Before Exam'].unique())

Unique Study Hours Before Exam values: ['3 – 4 hours' 'More than 6 hours' '5 – 6 hours' '1 – 2 hours'
 'Less than 1 hour']


In [68]:
hours_before_exam_map = {
    'Less than 1 hour' : 0.5,
    '1 – 2 hours'      : 1.5,
    '3 – 4 hours'      : 3.5,
    '5 – 6 hours'      : 5.5,
    'More than 6 hours': 7.0,
}

df['Study Hours Before Exam'] = (
    df['Study Hours Before Exam']
    .astype(str).str.strip()
    .map(hours_before_exam_map)
    .astype(float)
)
print(df['Study Hours Before Exam'].value_counts())
print('dtype:', df['Study Hours Before Exam'].dtype)

Study Hours Before Exam
7.0    49
3.5    27
5.5    22
1.5     3
0.5     2
Name: count, dtype: int64
dtype: float64


## 6. Procrastination Level — Already numeric, coerce to int

In [69]:
print('Unique Procrastination Level values:', df['Procrastination Level'].unique())

Unique Procrastination Level values: [2 3 5 4 1]


In [70]:
df['Procrastination Level'] = pd.to_numeric(df['Procrastination Level'], errors='coerce').astype('Int64')
print(df['Procrastination Level'].value_counts())
print('dtype:', df['Procrastination Level'].dtype)

Procrastination Level
3    42
4    34
5    14
2    12
1     1
Name: count, dtype: Int64
dtype: Int64


## 7. Work Hours per Week — Convert range strings to midpoint float

In [71]:
print('Unique Work Hours per Week values:', df['Work Hours per Week'].unique())

Unique Work Hours per Week values: [nan '5 – 10 hours' 'Less than 5 hours' '11 – 20 hours']


In [72]:
work_hours_map = {
    'Less than 5 hours': 2.5,
    '5 – 10 hours'     : 7.5,
    '11 – 20 hours'    : 15.5,
    'More than 20 hours': 25.0,
}

def convert_work_hours(val):
    val = str(val).strip()
    if val in work_hours_map:
        return work_hours_map[val]
    if val in ('', 'nan', 'NaN', 'No'):
        return 0.0    # non-working students = 0 hours
    try:
        return float(val)
    except ValueError:
        return np.nan

df['Work Hours per Week'] = df['Work Hours per Week'].apply(convert_work_hours).astype(float)
print(df['Work Hours per Week'].value_counts())
print('dtype:', df['Work Hours per Week'].dtype)

Work Hours per Week
0.0     92
7.5      5
2.5      4
15.5     2
Name: count, dtype: int64
dtype: float64


## 8. Sleep Hours — Convert range strings to midpoint float

In [73]:
print('Unique Sleep Hours values:', df['Sleep Hours'].unique())

Unique Sleep Hours values: ['6 – 7 hours' '4 – 5 hours' 'Less than 4 hours' '8 – 9 hours'
 'More than 9 hours']


In [74]:
sleep_map = {
    'Less than 4 hours': 3.0,
    '4 – 5 hours'      : 4.5,
    '6 – 7 hours'      : 6.5,
    '8 – 9 hours'      : 8.5,
    'More than 9 hours': 10.0,
}

df['Sleep Hours'] = (
    df['Sleep Hours']
    .astype(str).str.strip()
    .map(sleep_map)
    .astype(float)
)
print(df['Sleep Hours'].value_counts())
print('dtype:', df['Sleep Hours'].dtype)

Sleep Hours
4.5     45
6.5     40
8.5     10
3.0      7
10.0     1
Name: count, dtype: int64
dtype: float64


## 9 & 10. Stress Level (Beginning) and Stress Level (End) — Already numeric, coerce to int

In [75]:
print('Unique Stress Level (Beginning of Semester):', df['Stress Level (Beginning of Semester)'].unique())
print('Unique Stress Level (End of Semester):', df['Stress Level (End of Semester)'].unique())

Unique Stress Level (Beginning of Semester): [2 3 1 5 4]
Unique Stress Level (End of Semester): [4 5 3 1 2]


In [76]:
df['Stress Level (Beginning of Semester)'] = pd.to_numeric(df['Stress Level (Beginning of Semester)'], errors='coerce').astype('Int64')
df['Stress Level (End of Semester)']       = pd.to_numeric(df['Stress Level (End of Semester)'],       errors='coerce').astype('Int64')

print(df['Stress Level (Beginning of Semester)'].value_counts())
print(df['Stress Level (End of Semester)'].value_counts())

Stress Level (Beginning of Semester)
2    39
3    24
1    21
4    13
5     6
Name: count, dtype: Int64
Stress Level (End of Semester)
5    47
4    41
3     8
1     4
2     3
Name: count, dtype: Int64


## Final Check — Data Types & Missing Values

In [77]:
target_cols = [
    'CGPA',
    'Study Year',
    'Study Days per Week',
    'Study Hours per Day',
    'Study Hours Before Exam',
    'Procrastination Level',
    'Work Hours per Week',
    'Sleep Hours',
    'Stress Level (Beginning of Semester)',
    'Stress Level (End of Semester)',
]

summary = pd.DataFrame({
    'dtype'   : df[target_cols].dtypes,
    'missing' : df[target_cols].isna().sum(),
    'sample'  : df[target_cols].iloc[0],
})
print(summary)

                                        dtype  missing  sample
CGPA                                  float64        0     3.5
Study Year                              Int64        0     2.0
Study Days per Week                     Int64        0     6.0
Study Hours per Day                   float64        0     5.5
Study Hours Before Exam               float64        0     3.5
Procrastination Level                   Int64        0     2.0
Work Hours per Week                   float64        0     0.0
Sleep Hours                           float64        0     6.5
Stress Level (Beginning of Semester)    Int64        0     2.0
Stress Level (End of Semester)          Int64        0     4.0


In [78]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 21 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Timestamp                             103 non-null    object 
 1   Gender                                103 non-null    object 
 2   Course                                103 non-null    object 
 3   Study Year                            103 non-null    Int64  
 4   CGPA                                  103 non-null    float64
 5   Study Days per Week                   103 non-null    Int64  
 6   Study Hours per Day                   103 non-null    float64
 7   Study Hours Before Exam               103 non-null    float64
 8   Study Method                          103 non-null    object 
 9   Study Schedule                        103 non-null    object 
 10  Follow Schedule Frequency             36 non-null     object 
 11  Procrastination Lev

In [79]:
df.drop(columns=['Timestamp'], inplace=True)

In [80]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 20 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Gender                                103 non-null    object 
 1   Course                                103 non-null    object 
 2   Study Year                            103 non-null    Int64  
 3   CGPA                                  103 non-null    float64
 4   Study Days per Week                   103 non-null    Int64  
 5   Study Hours per Day                   103 non-null    float64
 6   Study Hours Before Exam               103 non-null    float64
 7   Study Method                          103 non-null    object 
 8   Study Schedule                        103 non-null    object 
 9   Follow Schedule Frequency             36 non-null     object 
 10  Procrastination Level                 103 non-null    Int64  
 11  Deadline Management

## Save Cleaned Dataset

In [81]:
df.to_csv('cleaned_dataset.csv', index=False)
print('Saved to cleaned_dataset.csv')

Saved to cleaned_dataset.csv


In [82]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [83]:
df.to_excel('Group 4_Cleaned Dataset_Student Academic Performance.xlsx', index=False)